# Qwen3-VL-4B — Authority condition ("Dr. Remy Ashford", verified)

Tests whether an authority cue in the poster's identity shifts competence or conformity.
Stimuli are the main study's Remy Ashford set regenerated with **only** the profile name and
verified badge changed — same 100 all-genres pie charts, same claims, same engagement values
(`utils/generate_dr_remy_posts.py`). Scope: **baseline + metrics/realistic only**.

Compare directly against this model's main-study results in `experiments/e1/qwen3-vl-4b/outputs/`;
the image sample is identical (`selected_images.json` copied from `experiments/e1/`).

Note: files are named `*_remy_ashford_*.png` (not `dr_`) because `e1_utils/sampling.py`
hardcodes that string — the condition is carried by the directory, not the filename.

In [ ]:
import sys, subprocess

subprocess.run([sys.executable, "-m", "pip", "uninstall", "torchaudio", "-y"])

subprocess.run([sys.executable, "-m", "pip", "install",
    "torch==2.6.0", "torchvision==0.21.0",
    "--index-url", "https://download.pytorch.org/whl/cu124",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "git+https://github.com/huggingface/transformers",
    "accelerate",
    "--user", "-q"], check=True)

subprocess.run([sys.executable, "-m", "pip", "install",
    "tokenizers>=0.22.0,<=0.23.0",
    "--user", "-q"], check=True)

print("✅ Done — restart the kernel now")

Restart kernel after running the setup cell above.

In [ ]:
!nvidia-smi

In [ ]:
import sys
sys.path.append("/home/jovyan")
from config_hf_token import HF_TOKEN
from huggingface_hub import login
login(token=HF_TOKEN)

from pathlib import Path
ROOT_DIR = Path().resolve().parents[2]
sys.path.insert(0, str(ROOT_DIR / "experiments/e1"))

import torch
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-4B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto"
)
print("✅ Loaded successfully")

In [ ]:
from e1_utils.sampling import build_paired_sample
from e1_utils.e1_optimized import (
    LIKE_PROMPT_SINGLE, LIKE_PROMPT_YESNO, LIKE_PROMPT_PAIR, ADJACENT_PAIRS,
    run_e1_baseline, run_e1_metrics, run_e1_metrics_paired
)
from e1_utils.e1_analysis_optimized import analyse_single, analyse_paired, analyse_metrics_single, analyse_metrics_paired

# --- Authority condition: "Dr. Remy Ashford" + verified badge ---
EXPERIMENT_DIR = Path().resolve().parent        # experiments/e1_authority/ -- shared sample across models
OUTPUT_DIR = Path().resolve() / "outputs"
SEED = 42
SAMPLE_SIZE = 100                                # matches the main study; selected_images.json is copied from experiments/e1/

correct_dir = ROOT_DIR / "benchmarking/correct/dr-remy-ashford"
incorrect_dir = ROOT_DIR / "benchmarking/incorrect/dr-remy-ashford"
all_images = build_paired_sample(correct_dir, incorrect_dir, SEED, SAMPLE_SIZE, EXPERIMENT_DIR)
selected_numbers = [name.replace("_correct", "") for name, _ in all_images if "_correct" in name]

# metrics/realistic only -- the condition with the strongest documented effect
correct_base = ROOT_DIR / "benchmarking/correct/dr-remy-ashford/metrics/realistic"
incorrect_base = ROOT_DIR / "benchmarking/incorrect/dr-remy-ashford/metrics/realistic"


In [ ]:
from e1_utils.inference_qwen import run_inference_qwen

In [ ]:
from e1_utils.e1_optimized import (
    run_e1_baseline_logprobs, run_e1_metrics_logprobs,
    LIKE_CANDIDATES_SINGLE, LIKE_CANDIDATES_YESNO
)

from e1_utils.inference_qwen import run_inference_with_scores_qwen

## Approach 1 -- single image, like/scroll, baseline (0 engagement)

In [ ]:
run_e1_baseline(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_baseline.json",
              inference_fn=run_inference_qwen)

In [ ]:
run_e1_baseline_logprobs(all_images, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_baseline_logprobs.json", score_fn=run_inference_with_scores_qwen)

In [ ]:
analyse_single(OUTPUT_DIR, "e1_results_baseline.json", like_answer="like")

## Approach 1 variant -- single image, like/scroll, across the 6 `metrics/realistic` engagement scales

In [ ]:
run_e1_metrics(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, output_filename="e1_results_metrics.json",
              inference_fn=run_inference_qwen)

In [ ]:
run_e1_metrics_logprobs(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR,
              prompt=LIKE_PROMPT_SINGLE, candidates=LIKE_CANDIDATES_SINGLE,
              output_filename="e1_results_metrics_logprobs.json", score_fn=run_inference_with_scores_qwen)

In [ ]:
analyse_metrics_single(OUTPUT_DIR, "e1_results_metrics.json", like_answer="like")

## Approach 2 -- paired A/B forced choice, full 7x7 `metrics/realistic` disparity grid

In [ ]:
import time
start = time.time()
run_e1_metrics_paired(selected_numbers, correct_base, incorrect_base, model, processor, device, OUTPUT_DIR, SEED,
              prompt=LIKE_PROMPT_PAIR, output_filename="e1_results_metrics_paired.json",
              inference_fn=run_inference_qwen)
elapsed = time.time() - start
print(f"\n⏱ Total runtime: {elapsed/60:.1f} min")

In [ ]:
analyse_metrics_paired(OUTPUT_DIR, "e1_results_metrics_paired.json")